# OCONUS NGWPC Hydrofabric Demo
This notebook walks through the following PI-9 acceptance criteria for the OCONUS (Alaska, Hawaii, Puerto Rico/Virgin Islands) domains. This should be run after building OCONUS NHF.

- domains include PRVI, Hawaii, and Alaska
- existence of the same river miles covered in the operational version of the NWM
- ensuring rivers can be represented as a directed acyclic graphs for routing
- connectivity checks
- flowpath and divide statistics (drainage area, length, etc)
- Ensure that hydrofabric fully complies with the HY_Features (WaterML2 Part 3) data model standard as extended to include flowlines that reduce artificial trans-basin transfers of flow.
- POIs should include at minimum all existing operational data assimilation gages as well as existing calibration gages and waterbodies (i.e. lake/reservoir) and their outlet nexuses.
- Every attempt will be made to maximize the NGWPC Hydrofabric such that the number of divides between 3-10 sq. km and verify that routing computational unit lengths (derived from flowpaths and flowlines) are an integer multiple of a 300 m discretization (acceptable range: 250-350 m)

In [1]:
from pathlib import Path

import geopandas as gpd
import pandas as pd

In [2]:
path_ak = Path("../data/ak_nhf_1.1.3.gpkg")
path_hi = Path("../data/hi_nhf_1.1.3.gpkg")
path_prvi =  Path("../data/prvi_nhf_1.1.3.gpkg")

pd.set_option("display.max_columns", None)

## Existence of the same river miles covered in the operational version of the NWM

TODO: open flowpaths for each layer and add to number of river miles in CONUS 

## Ensuring rivers can be represented as a directed acyclic graphs for routing / connectivity checks

TODO @Dylan: DAGs

## Flowpath and divide statistics (drainage area, length, etc)
Show the attributes available for divides and flowpath layers.

In [ ]:
def show_attributes(domain_path: Path, domain_name: str):
    """Display the fields in flowpaths and divides"""
    gdf_fp = gpd.read_file(domain_path, layer="flowpaths")
    print(f"{domain_name} Flowpath attributes")
    display(pd.DataFrame(data={"Columns":gdf_fp.columns}))

    gdf_div = gpd.read_file(domain_path, layer="divides")
    print(f"{domain_name} Divide Attributes")
    display(pd.DataFrame(data={"Columns":gdf_div.columns}))

In [11]:
# AK
show_attributes(path_ak, domain_name="AK")

AK Flowpath attributes


,Columns
0,fp_id
1,fp_to_id
2,dn_nex_id
3,up_nex_id
4,div_id
5,vpu_id
6,length_km
7,area_sqkm
8,total_da_sqkm
9,mainstem_lp


AK Divide Attributes


,Columns
0,div_id
1,vpu_id
2,type
3,area_sqkm
4,bexp_mode
5,isltyp_mode
6,ivgtyp_mode
7,dksat_geomean
8,psisat_geomean
9,cwpvt_mean


In [10]:
# HI
show_attributes(path_hi, domain_name="HI")

HI Flowpath attributes


,Columns
0,fp_id
1,fp_to_id
2,dn_nex_id
3,up_nex_id
4,div_id
5,vpu_id
6,length_km
7,area_sqkm
8,total_da_sqkm
9,mainstem_lp


HI Divide Attributes


,Columns
0,div_id
1,vpu_id
2,type
3,area_sqkm
4,bexp_mode
5,isltyp_mode
6,ivgtyp_mode
7,dksat_geomean
8,psisat_geomean
9,cwpvt_mean


In [12]:
# PRVI
show_attributes(path_hi, domain_name="PRVI")

PRVI Flowpath attributes


,Columns
0,fp_id
1,fp_to_id
2,dn_nex_id
3,up_nex_id
4,div_id
5,vpu_id
6,length_km
7,area_sqkm
8,total_da_sqkm
9,mainstem_lp


PRVI Divide Attributes


,Columns
0,div_id
1,vpu_id
2,type
3,area_sqkm
4,bexp_mode
5,isltyp_mode
6,ivgtyp_mode
7,dksat_geomean
8,psisat_geomean
9,cwpvt_mean


## Ensure that hydrofabric fully complies with the HY_Features (WaterML2 Part 3) data model standard as extended to include flowlines that reduce artificial trans-basin transfers of flow.
TODO @DYLAN Add text and go to OE link

## POIs should include at minimum all existing operational data assimilation gages as well as existing calibration gages and waterbodies (i.e. lake/reservoir) and their outlet nexuses

### Waterbodies / Lakes
The `lakes` layer was built in NHF to be a 1:1 representation of NWM operational waterbodies. The `lakes` layer retains all data from Hydrofabric 2.2 (Puerto Rico and Hawaii) and LAKEPARM (Alaska).

Where polygons were available (HI and PRVI), lakes were mapped to the most downstream intersecting flowpath. The most downstream flowpath is chosen with the minimum hydrosequence. In Alaska, points were mapped to their nearest flowpath.

In [17]:
def compare_lakes(nhf_path: Path, nwm_path: Path, domain: str, id_field: str):
    """Compare if lakes are present in an NWM source file and an NHF gages layer"""
    gdf_nwm = gpd.read_file(nwm_path)
    gdf_nhf = gpd.read_file(nhf_path, layer="lakes")
    print(f"{domain} NHF lakes: {len(gdf_nhf)}")
    print(f"{domain} NWM lakes: {len(gdf_nwm)}")
    print(f"{domain} lakes COMID in NWM: {len(gdf_nhf.loc[gdf_nhf['lake_id'].isin(gdf_nwm[id_field])])}")
    display(gdf_nhf.head())

Alaska lakes were retrieved from [NWM v3.0.18 LAKEPARM_AK.nc](https://www.nco.ncep.noaa.gov/pmb/codes/nwprod/nwm.v3.0.18/parm/domain_alaska/LAKEPARM_AK.nc) and saved to a GPKG.

In [18]:
compare_lakes(path_ak, "../data/lakes/input/ak_lakeparm.gpkg", "AK", "lake_id" )

AK NHF lakes: 237
AK NWM lakes: 237
AK lakes COMID in NWM: 237


,nhf_lake_id,ref_fp_id,hy_id,fp_id,virtual_fp_id,dn_nex_id,dn_virtual_nex_id,div_id,lake_id,res_id,LkArea,LkMxE,WeirC,WeirL,WeirE,OrificeC,OrificeA,OrificeE,Dam_Length,ifd,reservoir_index_AnA,reservoir_index_Extended_AnA,reservoir_index_GDL_AK,reservoir_index_Medium_Range,reservoir_index_Short_Range,geometry
0,1,810340249,317,171967.0,174539.0,171967.0,174516.0,171967.0,19029000019123,None,8.746114,242.112549,0.4,10.0,239.505161,0.1,1.0,224.729960,10.0,0.9,None,None,None,None,None,POINT (520915.823 1227965.773)
1,2,810434248,318,172875.0,177263.0,172875.0,177189.0,172875.0,19029000007860,None,67.990491,546.495728,0.4,10.0,545.103198,0.1,1.0,537.212199,10.0,0.9,None,None,None,None,None,POINT (424285.366 1327509.657)
2,3,810383934,319,149270.0,149606.0,149270.0,149586.0,149270.0,75004400013260,None,1.180562,65.401436,0.4,10.0,65.124396,0.1,1.0,63.554504,10.0,0.9,None,None,None,None,None,POINT (186823.192 1189116.815)
3,4,810252930,320,149349.0,149843.0,149349.0,149820.0,149349.0,75004400013261,None,1.034069,858.677734,0.4,10.0,856.423169,0.1,1.0,843.647298,10.0,0.9,None,None,None,None,None,POINT (213918.088 1142585.174)
4,5,810200869,321,119155.0,119172.0,119154.0,119164.0,119155.0,75004200016848,None,1.245905,55.758045,0.4,10.0,50.231239,0.1,1.0,18.912681,10.0,0.9,None,None,None,None,None,POINT (307586.841 1121446.442)


Hawaii lakes were extracted from `nwm_lakes.gpkg` provided by OWP in January 2026. The file was clipped to Hawaii state borders. `lake_id` and `newID` are both NHD COMID.

In [19]:
compare_lakes(path_hi, "../data/lakes/input/nwm_lakes_hi_input.gpkg", "HI", "newID")

HI NHF lakes: 9
HI NWM lakes: 9
HI lakes COMID in NWM: 9


,nhf_lake_id,ref_fp_id,hy_id,fp_id,virtual_fp_id,dn_nex_id,dn_virtual_nex_id,div_id,lake_id,res_id,LkArea,LkMxE,WeirC,WeirL,WeirE,OrificeC,OrificeA,OrificeE,Dam_Length,ifd,reservoir_index_AnA,reservoir_index_Extended_AnA,reservoir_index_GDL_AK,reservoir_index_Medium_Range,reservoir_index_Short_Range,geometry
0,1,80000600000848,426,11008.0,11102,11008,11094,11008,1,None,0.09,360.194092,0.4,10.0,354.923358,0.1,1.0,325.055868,10.0,0.9,NaN,NaN,NaN,NaN,NaN,POINT (605744.449 2377664.789)
1,2,80000600002192,427,10971.0,11165,10971,11053,10971,800021761,None,0.01,254.194519,0.4,10.0,253.494075,0.1,1.0,249.524892,10.0,0.9,NaN,NaN,NaN,NaN,NaN,POINT (596494.254 2382080.774)
2,3,80000500000193,428,8866.0,8869,8866,8868,8866,800021823,None,0.33,258.723114,0.4,10.0,257.683701,0.1,1.0,251.793691,10.0,0.9,NaN,NaN,NaN,NaN,NaN,POINT (702830.954 2340300.297)
3,4,80000700003811,429,16879.0,16886,16879,16883,16879,800022126,None,0.11,129.174454,0.4,10.0,127.928988,0.1,1.0,120.871348,10.0,0.9,NaN,NaN,NaN,NaN,NaN,POINT (458890.563 2453289.484)
4,5,80000200002063,430,4473.0,4558,4473,4557,4473,800022251,None,0.14,1.494061,0.4,10.0,1.395654,0.1,1.0,0.838020,10.0,0.9,NaN,NaN,NaN,NaN,NaN,POINT (762658.378 2301456.405)


Puerto Rico / Virgin Islands lakes were extracted from `nwm_lakes.gpkg` provided by OWP in January 2026. The file was clipped to Puerto Rico/Virgin Islands borders. `lake_id` and `newID` are both NHD COMID.

In [20]:
compare_lakes(path_prvi, "../data/lakes/input/nwm_lakes_prvi_input.gpkg", "PRVI", "newID")

PRVI NHF lakes: 15
PRVI NWM lakes: 15
PRVI lakes COMID in NWM: 15


,nhf_lake_id,ref_fp_id,hy_id,fp_id,virtual_fp_id,dn_nex_id,dn_virtual_nex_id,div_id,lake_id,res_id,LkArea,LkMxE,WeirC,WeirL,WeirE,OrificeC,OrificeA,OrificeE,Dam_Length,ifd,reservoir_index_AnA,reservoir_index_Extended_AnA,reservoir_index_GDL_AK,reservoir_index_Medium_Range,reservoir_index_Short_Range,geometry
0,1,85000100016408,246,20915.0,21303,20915,21503,20915,800042956,None,1.078000,408.250641,0.4,10.0,406.994379,0.1,1.0,399.875559,10.0,0.9,NaN,NaN,NaN,NaN,NaN,POINT (231121.425 239265.659)
1,2,85000100003398,247,5325.0,5485,5325,5484,5325,800043166,None,1.404556,110.557816,0.4,10.0,108.301464,0.1,1.0,95.515472,10.0,0.9,NaN,NaN,NaN,NaN,NaN,POINT (192675.867 228922.114)
2,3,85000100013084,248,15726.0,16738,15726,16141,15726,800043413,None,2.343625,259.772095,0.4,10.0,254.799779,0.1,1.0,226.623322,10.0,0.9,NaN,NaN,NaN,NaN,NaN,POINT (176779.105 247208.96)
3,4,85000100012414,249,15704.0,16788,15704,16763,15704,800043415,None,1.670567,102.502541,0.4,10.0,97.507730,0.1,1.0,69.203801,10.0,0.9,NaN,NaN,NaN,NaN,NaN,POINT (175807.217 254316.217)
4,5,85000100010260,250,18219.0,18306,18219,18270,18219,800043881,None,5.475250,1.456729,0.4,10.0,1.320885,0.1,1.0,0.551099,10.0,0.9,NaN,NaN,NaN,NaN,NaN,POINT (242976.857 265766.678)


### Gages
Gages were extracted from routelink and USGS. Routelink files were downloaded from NWM v3.0.18, converted to GPKG in EPSG:4326, and extracted any row with a populated gage ID field. 

If upstream area information was available, gages were matched to flowpaths/divides using it. If it was not available, gages were matched to nearest flowpath. See connectivity columns in tables below (fp_id, virtual_fp_id, dn_nex_id, dn_virtual_nex_id).

In [13]:
def compare_gages(nhf_path: Path, routelink_path: Path, domain: str):
    """Compare if gages are present in routelink and an NHF gages layer"""
    gdf_routelink = gpd.read_file(routelink_path)

    # extract rows with gages
    gdf_routelink["gages"] = gdf_routelink["gages"].str.strip()
    gdf_routelink = gdf_routelink.loc[gdf_routelink["gages"] != ""].copy()

    gdf_gages = gpd.read_file(nhf_path, layer="gages")
    print(f"{domain} NHF gages: {len(gdf_gages)}")
    print(f"{domain} Routelink gages: {len(gdf_routelink)}")
    print(f"{domain} gage ID in Routelink: {len(gdf_gages.loc[gdf_gages['site_no'].isin(gdf_routelink['gages'])])}")
    display(gdf_gages.head())

In [ ]:
# Function used in NHF-builds to extract routelink gages - this is for demonstration purposes only
def append_from_routelink(
    gdf: gpd.GeoDataFrame, routelink: Path, id_col_name: str, shape: Path | None
) -> gpd.GeoDataFrame:
    """Append gages from RouteLink file to GeoDataFrame

    Use ogr2ogr to convert NC file to GPKG and add EPSG:4326 georef i.e. ogr2ogr RouteLink.gpkg RouteLink.nc -t_srs EPSG:4326 -s_srs EPSG:4326

    Parameters
    ----------
    gdf: GeoDataFrame
        Input dataframe to append to
    routelink : Path
        RouteLink file to extract from
    id_col_name: str
        Column to pull from for site_no in RouteLink
    shape: Path | None
        Shapefile to use for clipping
    """
    gages = gpd.read_file(routelink).to_crs(gdf.crs)

    # first get gages only
    gages = gages.loc[gages[id_col_name].str.strip() != ""].copy()

    # then check intersection if requested
    if shape:
        # Get boundary to clip to
        shp = gpd.read_file(shape).to_crs(gdf.crs)
        merged_geom = shp["geometry"].union_all()
        gages = gages.loc[gages["geometry"].intersects(merged_geom), :].copy()

    gages = gages.rename(columns={id_col_name: "site_no"})
    gages["site_no"] = gages["site_no"].str.strip()

    gages = gpd.GeoDataFrame(gages[["geometry", "site_no"]][~gages["site_no"].isin(gdf["site_no"])].copy())
    # logger.info(f"gages: added {len(gages)} gages from RouteLink not already present in dataset") # commetned for missing imports in demonstration
    gages["status"] = "routelink"
    gages = pd.concat([gdf, gages])
    gages["geometry"] = gages["geometry"].force_2d()

    return gages

**Alaska Note**: There are some missing gages in Alaska due to a domain mismatch between the NHF AK reference and the NWM AK domain. The GEOGLOWS dataset used for the reference does not include some coastal areas of AK. This domain problem will be rectified in the next version of AK domain.

In [ ]:
# AK
compare_gages(path_ak, Path("../data/gages/routelink/RouteLink_AK_EPSG4326.gpkg"), "AK")

AK NHF gages: 316
AK Routelink gages: 65
AK gage ID in Routelink: 44


,site_no,status,hy_id,USGS_basin_km2,ref_fp_id,method_fp_to_gage,fp_id,virtual_fp_id,mainstem_virtual_fp_id,segment_order,div_id,dn_nex_id,dn_virtual_nex_id,geometry
0,15008000,USGS-active,1,NaN,720093657,nearest_fp,7688,7735,7735,0,7688,7688,7727,POINT (1458336.94 934072.682)
1,15010000,USGS-discontinued,2,NaN,720079798,nearest_fp,1330,1341,1341,0,1330,1330,1340,POINT (1461278.366 902279.005)
2,15011500,USGS-discontinued,3,NaN,720109625,nearest_fp,12603,12638,12638,0,12603,12603,12633,POINT (1465303.241 831444.434)
3,15011870,USGS-discontinued,4,NaN,720063962,nearest_fp,6837,6871,6871,0,6837,6836,6863,POINT (1458945.907 861482.545)
4,15011880,USGS-discontinued,5,NaN,720109600,nearest_fp,6834,6862,6862,0,6834,6834,6854,POINT (1461980.525 855413.614)


In [15]:
# HI
compare_gages(path_hi, Path("../data/gages/routelink/RouteLink_HI_EPSG4326.gpkg"), "Hawaii")

Hawaii NHF gages: 425
Hawaii Routelink gages: 58
Hawaii gage ID in Routelink: 58


,site_no,status,hy_id,USGS_basin_km2,ref_fp_id,method_fp_to_gage,fp_id,virtual_fp_id,mainstem_virtual_fp_id,segment_order,div_id,dn_nex_id,dn_virtual_nex_id,geometry
0,16010000,USGS-active,1,NaN,80000700000159,nearest_fp,19704.0,20105,20105,0,19704,19704.0,19777,POINT (436067.669 2447657.104)
1,16011000,USGS-discontinued,2,NaN,80000700002925,nearest_fp,19709.0,20207,20207,2,19709,19708.0,20465,POINT (435917.283 2446813.545)
2,16012000,USGS-discontinued,3,NaN,80000700001275,nearest_fp,19698.0,19971,19971,2,19698,19697.0,19938,POINT (435033.001 2447739.395)
3,16013000,USGS-discontinued,4,NaN,80000700002922,nearest_fp,19684.0,20434,20434,0,19684,19683.0,20094,POINT (438005.105 2445913.214)
4,16014000,USGS-discontinued,5,NaN,80000700000650,nearest_fp,13913.0,13930,13930,0,13913,13913.0,13919,POINT (430293.769 2444994.348)


In [16]:
# PRVI
compare_gages(path_prvi, Path("../data/gages/routelink/RouteLink_PRVI_EPSG4326.gpkg"), "PRVI")

PRVI NHF gages: 245
PRVI Routelink gages: 83
PRVI gage ID in Routelink: 82


,site_no,status,hy_id,USGS_basin_km2,ref_fp_id,method_fp_to_gage,fp_id,virtual_fp_id,mainstem_virtual_fp_id,segment_order,div_id,dn_nex_id,dn_virtual_nex_id,geometry
0,180157066305000,USGS-discontinued,1,NaN,85000100004561,nearest_fp,6409.0,6418,6418,1,6409,6409.0,6416,POINT (191746.183 221949.368)
1,180200066304200,USGS-discontinued,2,NaN,85000100004561,nearest_fp,6409.0,6418,6418,1,6409,6409.0,6416,POINT (191540.233 221826.486)
2,181019066091300,USGS-discontinued,3,NaN,85000100016274,nearest_fp,20923.0,21261,21261,1,20923,20923.0,21285,POINT (229636.443 237280.585)
3,181023066095700,USGS-discontinued,4,NaN,85000100004398,nearest_fp,6569.0,7279,7279,1,6569,6568.0,7926,POINT (228343.104 237401.623)
4,181042066091900,USGS-discontinued,5,NaN,85000100016493,nearest_fp,NaN,21012,21287,0,20920,NaN,21012,POINT (229459.027 237987.418)


## Gage / Lake Flowpath Association
TODO @QUERCUS
demonstrate with figures how a point is matched to flowpath/nexus

## Every attempt will be made to maximize the NGWPC Hydrofabric such that the number of divides between 3-10 sq. km and verify that routing computational unit lengths (derived from flowpaths and flowlines) are an integer multiple of a 300 m discretization (acceptable range: 250-350 m)
TODO @DYLAN